# 02 · Bronze → Silver | Acidentes ANTT

| | |
|---|---|
| **Origem** | `Files/bronze/acidentes/*.csv` |
| **Destino** | Delta Table `silver_acidentes` |
| **Partição** | `ano`, `concessionaria` |
| **Idempotente** | Sim — `overwrite` recria a tabela |

**Transformações aplicadas:** encoding Windows-1252 · parse de data/km · normalização de sentido e tipo de ocorrência · derivação de concessionária · split de trecho · colunas derivadas (total_veiculos, total_vitimas, severidade)

## 2 · Parâmetros

> Célula marcada como **Parameters** — valores podem ser sobrescritos por um Data Pipeline.

In [ ]:
NOTEBOOK_NAME:  str  = "bronze_to_silver"
ORIGEM_BRONZE:  str  = "Files/bronze/acidentes"
TABELA_DESTINO: str  = "silver_acidentes"
MODO_ESCRITA:   str  = "overwrite"           # overwrite | append
PARTICOES:      list = ["ano", "concessionaria"]
LOG_LEVEL:      str  = "INFO"

## 3 · Configuração Compartilhada

In [ ]:
%run 00_nb_config

In [ ]:
%run 00_nb_helpers

## 4 · Execução

In [ ]:
df_bronze = ler_bronze(ORIGEM_BRONZE)
df_silver = transformar(mapear_concessionaria(df_bronze, CONCESSAO_MAP))
validar(df_silver)
salvar_silver(df_silver, TABELA_DESTINO, MODO_ESCRITA, PARTICOES)

## 5 · Relatório

In [ ]:
log.info("=== Schema da tabela %s ===", TABELA_DESTINO)
spark.sql(f"DESCRIBE TABLE {TABELA_DESTINO}").show(60, truncate=False)

log.info("=== Registros por concessionária ===")
spark.sql(f"""
    SELECT concessionaria, COUNT(*) AS registros
    FROM {TABELA_DESTINO}
    GROUP BY concessionaria
    ORDER BY registros DESC
""").show(40, truncate=False)

notebookutils.notebook.exit(f"OK: tabela {TABELA_DESTINO} criada com sucesso.")